In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3:14b", temperature=0)

In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(state: dict) -> str:
    """Read the email from the current state. ALWAYS call this first."""
    return state.get("email", "No email found")

@tool
def send_email(body: str, recipient: str = "John") -> str:
    """Send an email response. Requires human approval before execution."""
    return f"✅ Email sent to {recipient}: {body}"

In [6]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailAgentState(AgentState):
    messages: Annotated[list, add_messages]
    email: str

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailAgentState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,   # No interrupt for reading
                "send_email": True,    # Interrupt before sending
            },
            description_prefix="⚠️ Tool execution requires approval",
        ),
    ],
)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

config = {"configurable": {"thread_id": "1"}}

input_data = {
    "messages": [
        SystemMessage(content="You are an email assistant. Read the email first, then draft and send a response."),
        HumanMessage(content="Please read my email and send a response.")
    ],
    "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
}

print("🚀 Starting agent execution...\n")

for event in agent.stream(input_data, config=config, stream_mode="updates"):
    pprint(event)
    print("-" * 60)

🚀 Starting agent execution...



In [ ]:
state = agent.get_state(config)
interrupts = state.metadata.get("interrupts", [])

if interrupts:
    print("\n⚠️  EXECUTION INTERRUPTED!")
    print(f"Interrupt details: {interrupts}")
    
    print("\n📋 Current State:")
    pprint(state.values)
    
    # Resume execution after human approval
    print("\n✅ Resuming execution after approval...\n")
    for event in agent.stream(None, config=config, stream_mode="updates"):
        pprint(event)
        print("-" * 60)
else:
    print("\n✅ Execution completed without interrupts")

In [ ]:
final_state = agent.get_state(config)
print("\n📬 FINAL STATE:")
pprint(final_state.values)

# Show all messages
print("\n💬 CONVERSATION:")
for msg in final_state.values.get("messages", []):
    print(f"{msg.type}: {msg.content[:150]}...")

In [ ]:
print(response['__interrupt__'])

In [ ]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Approve

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

## Reject

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

In [ ]:
response['messages'][-3].tool_calls[0]['args']['body']

## Edit

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)